# SU(3) O(y⁵)/O(y⁶) historical coefficient recovery
Run all cells. The notebook searches `/content`, mounted Drive, and `/mnt/data`; if source PDFs are missing, it opens one upload picker. No code edits are required.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pymupdf', 'sympy', 'pytesseract', 'pillow'])
print('dependencies ready')

In [ ]:
from pathlib import Path
SOURCE = '#!/usr/bin/env python3\n"""\nSU(3) O(y^5)/O(y^6) historical coefficient recovery.\n\nPurpose\n-------\nRecover the fifth- and sixth-order Hamiltonian strong-coupling inputs from\nthe historical source PDFs, with exact normalization checks against the\nalready-proved project coefficients through O(y^4).\n\nThe script searches /content, mounted Google Drive, and /mnt/data.  If the\npapers are not present in Colab it opens one upload picker.  It then:\n\n  * extracts text page-by-page (PyMuPDF);\n  * uses OCR only when a PDF is image-only;\n  * ranks pages containing series/coefficient keywords;\n  * records equation-like lines and candidate numeric tables;\n  * prints the exact coupling conversion and coefficient fingerprints;\n  * emits a Markdown and JSON extraction ledger.\n\nNo historical coefficient is accepted automatically unless its convention\nmatches the known project fingerprint through O(y^4).\n\nCorrect normalization\n---------------------\nProject:\n  H_beta = 1/2 sum C2 + beta sum_p (1 - ReTr(U_p)/3),\n  y = 2 beta / 3.\n\nStandard dimensionless Hamiltonian:\n  H_tilde = 1/2 sum E^2 - lambda/6 sum_p Tr(U_p+U_p^\\dagger),\n  lambda = 6/g_H^4.\n\nAfter dropping the plaquette constant:\n  beta = lambda,\n  y = 2 lambda / 3 = 4/g_H^4,\n  lambda = 3y/2.\n\nTherefore, if F(lambda)=sum a_n lambda^n, then the project-y coefficient is\n  f_n = a_n (3/2)^n,\nor equivalently\n  a_n = f_n (2/3)^n.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport re\nimport subprocess\nimport sys\nfrom pathlib import Path\nfrom typing import Any\n\n# Install imports only when needed.\ntry:\n    import fitz  # PyMuPDF\nexcept Exception:\n    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pymupdf"])\n    import fitz\n\ntry:\n    import sympy as sp\nexcept Exception:\n    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sympy"])\n    import sympy as sp\n\nBASE = Path("/content") if Path("/content").exists() else Path("/mnt/data")\nOUT = BASE / "SU3_Y5_Y6_HISTORICAL_RECOVERY"\nOUT.mkdir(parents=True, exist_ok=True)\n\nSOURCES = {\n    "KPS_1981_string": {\n        "title": "The String Tension, Confinement and Roughening in SU(3) Hamiltonian Lattice Gauge Theory",\n        "authors": "J. B. Kogut, R. B. Pearson, J. Shigemitsu",\n        "doi": "10.1016/0370-2693(81)90369-5",\n        "expected_pages": "6",\n        "purpose": "string tension through O(g_H^-24), hence through y^6",\n        "filename_hints": ["kogut", "pearson", "shigemitsu", "90369", "string_tension"],\n    },\n    "HIP_1986_string": {\n        "title": "Cluster expansion approach to non-abelian lattice gauge theory in (3+1)D (II). SU(3)",\n        "authors": "C. J. Hamer, A. C. Irving, T. E. Preece",\n        "citation": "Nucl. Phys. B270 (1986) 553",\n        "purpose": "exact linked-cluster axial string-tension series and independent check",\n        "filename_hints": ["hamer", "irving", "preece", "b270", "cluster", "su3"],\n    },\n    "HAMER_1989_mass": {\n        "title": "Hamiltonian strong coupling expansions for glueball masses in SU(3)",\n        "authors": "C. J. Hamer",\n        "doi": "10.1016/0370-2693(89)91242-2",\n        "expected_pages": "4",\n        "purpose": "0++, 1+-, 2++ glueball mass series; candidate source for m5,m6",\n        "filename_hints": ["hamer", "91242", "glueball", "mass"],\n    },\n}\n\n# Exact project coefficients.\nm4 = -sp.Rational(20721577909065127111, 7250590288602460800)\nsigma4 = -sp.Rational(246334830054896087, 2788688572539408000)\n\nM_Y = [\n    sp.Rational(8, 3),\n    sp.Rational(1),\n    sp.Rational(11, 306),\n    -sp.Rational(109151, 249696),\n    m4,\n]\nSIGMA_Y = [\n    sp.Rational(2, 3),\n    sp.Rational(0),\n    -sp.Rational(22, 153),\n    sp.Rational(0),\n    sigma4,\n]\n\ndef transform_coeffs_y_to_lambda(coeffs):\n    # lambda = 3y/2 => y = 2lambda/3\n    return [sp.factor(c * sp.Rational(2, 3)**n) for n, c in enumerate(coeffs)]\n\ndef transform_coeffs_y_to_x(coeffs):\n    # x = g_H^-4, y = 4x.\n    return [sp.factor(c * 4**n) for n, c in enumerate(coeffs)]\n\nM_LAMBDA = transform_coeffs_y_to_lambda(M_Y)\nSIGMA_LAMBDA = transform_coeffs_y_to_lambda(SIGMA_Y)\nM_X = transform_coeffs_y_to_x(M_Y)\nSIGMA_X = transform_coeffs_y_to_x(SIGMA_Y)\n\ndef sha256(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open("rb") as f:\n        for b in iter(lambda: f.read(1 << 20), b""):\n            h.update(b)\n    return h.hexdigest()\n\ndef candidate_roots():\n    roots = [BASE, Path("/mnt/data")]\n    drive = Path("/content/drive")\n    if drive.exists():\n        roots.append(drive)\n    return roots\n\ndef all_pdfs():\n    seen = {}\n    for root in candidate_roots():\n        if not root.exists():\n            continue\n        try:\n            for p in root.rglob("*.pdf"):\n                if p.is_file():\n                    try:\n                        seen[sha256(p)] = p\n                    except Exception:\n                        pass\n        except Exception:\n            pass\n    return list(seen.values())\n\ndef score_filename(path: Path, hints: list[str]) -> int:\n    name = path.name.lower()\n    return sum(4 for h in hints if h.lower() in name)\n\ndef first_page_text(path: Path) -> str:\n    try:\n        doc = fitz.open(path)\n        return doc[0].get_text("text")[:6000] if len(doc) else ""\n    except Exception:\n        return ""\n\ndef score_content(text: str, source: dict[str, Any]) -> int:\n    t = text.lower()\n    score = 0\n    for word in re.findall(r"[a-z0-9]+", source["title"].lower()):\n        if len(word) >= 4 and word in t:\n            score += 1\n    for author in re.findall(r"[a-z]+", source["authors"].lower()):\n        if len(author) >= 5 and author in t:\n            score += 2\n    return score\n\ndef locate_sources():\n    pdfs = all_pdfs()\n    selected = {}\n    for key, source in SOURCES.items():\n        ranked = []\n        for p in pdfs:\n            ft = first_page_text(p)\n            score = score_filename(p, source["filename_hints"]) + score_content(ft, source)\n            if score:\n                ranked.append((score, p))\n        ranked.sort(key=lambda x: (-x[0], str(x[1])))\n        if ranked:\n            selected[key] = ranked[0][1]\n    return selected\n\ndef upload_missing(missing: list[str]):\n    if not missing or not Path("/content").exists():\n        return\n    try:\n        from google.colab import files  # type: ignore\n    except Exception:\n        return\n    print("\\nPDF UPLOAD REQUIRED")\n    print("Select any of the following source PDFs that you can access:")\n    for key in missing:\n        s = SOURCES[key]\n        print(f"  - {s[\'authors\']}: {s[\'title\']}")\n    uploaded = files.upload()\n    for name, data in uploaded.items():\n        target = Path("/content") / Path(name).name\n        target.write_bytes(data)\n        print(f"saved {len(data):,} bytes -> {target}")\n\nKEYWORDS = [\n    "1+-", "1 +-", "1+−", "1^{+-}", "glueball", "mass",\n    "string tension", "axial", "series", "strong coupling",\n    "lambda", "g^-", "g−", "order", "table", "pade", "roughening",\n]\n\ndef ocr_page(page) -> str:\n    """OCR one page only when embedded text is absent."""\n    try:\n        import pytesseract\n        from PIL import Image\n    except Exception:\n        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pytesseract", "pillow"])\n        import pytesseract\n        from PIL import Image\n    # Install tesseract binary in Colab only when absent.\n    if subprocess.call(["bash", "-lc", "command -v tesseract >/dev/null 2>&1"]) != 0:\n        subprocess.check_call(["bash", "-lc", "apt-get update -qq && apt-get install -y -qq tesseract-ocr"])\n    pix = page.get_pixmap(matrix=fitz.Matrix(2.5, 2.5), alpha=False)\n    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)\n    return pytesseract.image_to_string(img)\n\ndef extract_pages(path: Path):\n    doc = fitz.open(path)\n    pages = []\n    raw_lengths = []\n    for i, page in enumerate(doc):\n        text = page.get_text("text")\n        raw_lengths.append(len(text.strip()))\n        pages.append(text)\n    image_only = (sum(raw_lengths) / max(1, len(raw_lengths))) < 120\n    if image_only:\n        print(f"OCR fallback: {path.name} appears image-only ({len(doc)} pages)")\n        pages = [ocr_page(page) for page in doc]\n    return pages, image_only\n\ndef page_score(text: str) -> int:\n    t = text.lower()\n    score = 0\n    for kw in KEYWORDS:\n        score += 3 * t.count(kw.lower())\n    score += 2 * len(re.findall(r"(?:=|≈|~)\\s*[+\\-]?\\d", text))\n    score += len(re.findall(r"\\bg\\s*[\\-\\^−]*\\s*\\d+", t))\n    return score\n\ndef clean_line(line: str) -> str:\n    return re.sub(r"\\s+", " ", line).strip()\n\ndef equation_like_lines(text: str):\n    out = []\n    for line in text.splitlines():\n        c = clean_line(line)\n        if len(c) < 6 or len(c) > 240:\n            continue\n        has_num = bool(re.search(r"\\d", c))\n        has_series = any(k in c.lower() for k in ["lambda", "g", "mass", "tension", "series", "order", "1+-"])\n        has_eq = any(s in c for s in ["=", "≈", "~", "+", "−", "-"])\n        if has_num and has_series and has_eq:\n            out.append(c)\n    return out[:120]\n\ndef fingerprint_markdown(label: str, coeffs_y, coeffs_lam, coeffs_x):\n    lines = [f"### {label}", "", "| n | project y coefficient | lambda coefficient | x=g_H^-4 coefficient |",\n             "|---:|---:|---:|---:|"]\n    for n in range(len(coeffs_y)):\n        lines.append(f"| {n} | `{coeffs_y[n]}` | `{coeffs_lam[n]}` | `{coeffs_x[n]}` |")\n    return "\\n".join(lines)\n\ndef main():\n    print("=" * 112)\n    print("SU(3) O(y^5)/O(y^6) HISTORICAL COEFFICIENT RECOVERY")\n    print("=" * 112)\n    print("Correct conversion: beta=lambda=6/g_H^4, y=2lambda/3=4/g_H^4, lambda=3y/2")\n    print()\n\n    located = locate_sources()\n    missing = [k for k in SOURCES if k not in located]\n    upload_missing(missing)\n    located = locate_sources()\n    missing = [k for k in SOURCES if k not in located]\n\n    records = {}\n    for key, path in located.items():\n        print(f"\\nPROCESSING {key}: {path}")\n        pages, used_ocr = extract_pages(path)\n        ranked = sorted([(page_score(t), i, t) for i, t in enumerate(pages)], reverse=True)\n        top = []\n        for score, i, text in ranked[:8]:\n            top.append({\n                "page": i + 1,\n                "score": score,\n                "snippet": clean_line(text[:3500]),\n                "equation_like_lines": equation_like_lines(text),\n            })\n        records[key] = {\n            "source": SOURCES[key],\n            "path": str(path),\n            "sha256": sha256(path),\n            "pages": len(pages),\n            "ocr_used": used_ocr,\n            "top_pages": top,\n        }\n        print("top pages:", [(x["page"], x["score"]) for x in top])\n\n    status = "SOURCE_PDFS_INCOMPLETE" if missing else "ALL_SOURCE_PDFS_PARSED"\n    payload = {\n        "status": status,\n        "normalization": {\n            "project": "H_beta=1/2 sum C2 + beta sum(1-ReTr/3); y=2 beta/3",\n            "historical": "H_tilde=1/2 sum E^2 - lambda/6 sum Tr(U+Udag); lambda=6/g_H^4",\n            "conversion": "beta=lambda; y=2lambda/3=4/g_H^4; lambda=3y/2",\n        },\n        "known_fingerprints": {\n            "mass_y": [str(x) for x in M_Y],\n            "mass_lambda": [str(x) for x in M_LAMBDA],\n            "mass_x_gminus4": [str(x) for x in M_X],\n            "sigma_y": [str(x) for x in SIGMA_Y],\n            "sigma_lambda": [str(x) for x in SIGMA_LAMBDA],\n            "sigma_x_gminus4": [str(x) for x in SIGMA_X],\n        },\n        "located": records,\n        "missing": {k: SOURCES[k] for k in missing},\n        "acceptance_rule": (\n            "Do not accept m5,m6,sigma5,sigma6 until the source convention reproduces "\n            "the known coefficients through y^4 after exact conversion."\n        ),\n    }\n\n    json_path = OUT / "SU3_Y5_Y6_HISTORICAL_RECOVERY.json"\n    json_path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")\n\n    md = [\n        "# SU(3) O(y^5)/O(y^6) historical coefficient recovery",\n        "",\n        f"**Status:** `{status}`",\n        "",\n        "## Correct coupling normalization",\n        "",\n        r"\\[",\n        r"H_\\beta=\\frac12\\sum C_2+\\beta\\sum_p\\left(1-\\frac13\\operatorname{ReTr}U_p\\right),",\n        r"\\qquad y=\\frac{2\\beta}{3}.",\n        r"\\]",\n        "",\n        r"The standard dimensionless Hamiltonian uses",\n        "",\n        r"\\[",\n        r"\\widetilde H=\\frac12\\sum E^2-\\frac{\\lambda}{6}\\sum_p\\operatorname{Tr}(U_p+U_p^\\dagger),",\n        r"\\qquad \\lambda=\\frac6{g_H^4}.",\n        r"\\]",\n        "",\n        r"After dropping the plaquette constant, \\(\\beta=\\lambda\\). Therefore",\n        "",\n        r"\\[",\n        r"\\boxed{y=\\frac{2\\lambda}{3}=\\frac4{g_H^4}},",\n        r"\\qquad",\n        r"\\boxed{\\lambda=\\frac{3y}{2}}.",\n        r"\\]",\n        "",\n        "This corrects the previously stated conversion `y=2/g_H^4`.",\n        "",\n        "## Exact fingerprints",\n        "",\n        fingerprint_markdown("1+- glueball mass", M_Y, M_LAMBDA, M_X),\n        "",\n        fingerprint_markdown("axial string tension", SIGMA_Y, SIGMA_LAMBDA, SIGMA_X),\n        "",\n        "## Source extraction",\n        "",\n    ]\n    for key in SOURCES:\n        if key not in records:\n            md += [\n                f"### {key}",\n                "",\n                "**Missing PDF.**",\n                "",\n                f"- {SOURCES[key][\'authors\']}",\n                f"- *{SOURCES[key][\'title\']}*",\n                f"- purpose: {SOURCES[key][\'purpose\']}",\n                "",\n            ]\n            continue\n        rec = records[key]\n        md += [\n            f"### {key}",\n            "",\n            f"- file: `{rec[\'path\']}`",\n            f"- SHA-256: `{rec[\'sha256\']}`",\n            f"- pages: {rec[\'pages\']}",\n            f"- OCR used: {rec[\'ocr_used\']}",\n            "",\n            "Highest-scoring pages:",\n            "",\n        ]\n        for p in rec["top_pages"]:\n            md += [\n                f"#### Page {p[\'page\']} — score {p[\'score\']}",\n                "",\n                "```text",\n                p["snippet"][:2000],\n                "```",\n                "",\n            ]\n            if p["equation_like_lines"]:\n                md += ["Candidate equation/table lines:", "", "```text"]\n                md += p["equation_like_lines"][:40]\n                md += ["```", ""]\n\n    md += [\n        "## Acceptance gate",\n        "",\n        "A historical coefficient table is accepted only if exact conversion reproduces every known",\n        "coefficient through fourth order.  Only then are its fifth- and sixth-order entries promoted",\n        "to provisional project inputs.",\n        "",\n        "The script does not fabricate coefficients when a source PDF is missing or ambiguous.",\n    ]\n\n    md_path = OUT / "SU3_Y5_Y6_HISTORICAL_RECOVERY.md"\n    md_path.write_text("\\n".join(md), encoding="utf-8")\n\n    print("\\nSTATUS:", status)\n    print("JSON:", json_path)\n    print("MD:  ", md_path)\n    if missing:\n        print("\\nMISSING SOURCE PDFS:")\n        for key in missing:\n            print(" -", SOURCES[key]["authors"], "—", SOURCES[key]["title"])\n    print("=" * 112)\n\nif __name__ == "__main__":\n    main()\n'
path = Path('/content/su3_y5_y6_historical_recovery.py')
path.write_text(SOURCE, encoding='utf-8')
exec(compile(SOURCE, str(path), 'exec'), {'__name__': '__main__'})

In [ ]:
from pathlib import Path
import zipfile
out = Path('/content/SU3_Y5_Y6_HISTORICAL_RECOVERY')
bundle = Path('/content/SU3_Y5_Y6_HISTORICAL_RECOVERY_RESULTS.zip')
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in out.rglob('*'):
        if p.is_file(): zf.write(p, p.relative_to(out.parent))
print('RESULT BUNDLE:', bundle)